# Check the prod_AuAu_0_10 output

Reads the `.h5` files written by `run_prod.py` / `run_jobs.sh` and checks them:

1. **Summary**: grid, per-event frame count, freeze-out time, `tau0`, wall time; file-level consistency.
2. **Sanity scan**: every written frame is finite, has $\varepsilon \ge 0$ and $|v| < 1$; frames past `ntau_freezeout` are exactly zero.
3. **Evolution vs $\tau$**: mean and maximum $\varepsilon$, and the $\varepsilon$-weighted transverse flow $\langle v_T\rangle_\varepsilon$.
4. **Freeze-out time distribution** over all events.
5. **$x$–$y$ viewer**: $\varepsilon$, $v_x$, $v_y$, $v_z$ at one $\eta_s$ slice, with sliders for event, $\tau$ and $\eta_s$.

Only `numpy`, `h5py`, `matplotlib`, `pandas` and `ipywidgets` are needed (no X-SCAPE build), so it runs on
any machine that has the files. Set `FILES` below, or the `PROD_H5_GLOB` environment variable.

Layout of each file: `arr` is `(nevents, 4, nx, ny, neta, ntau)` float32 with channels
`energy_density [GeV/fm^3], vx, vy, vz` (Cartesian lab velocities), stored one $\tau$ frame per HDF5 chunk,
so reading one frame is cheap. `ntau_freezeout[i]` is the number of frames written for event `i`, and
`tau_freezeout[i]` is when the hydro ended (taken from MUSIC's own grid).

In [ ]:
%matplotlib inline
import glob
import os

import h5py
try:  # registers Blosc, the default compression since h5_optim (../../README_h5_optim.md)
    import hdf5plugin  # noqa: F401
except ImportError:
    print("note: pip install hdf5plugin -- needed for Blosc-compressed files (lzf files read without it)")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize

# Files to check (a glob, relative to this notebook; run_jobs.sh writes ./out by default).
FILES = os.environ.get("PROD_H5_GLOB", "out/AuAu_0_10_seed*.h5")
# The frame-by-frame scan (sections 2-3) reads every written frame: ~0.2 GB per event.
# Scan at most this many events (None = all).
MAX_SCAN_EVENTS = 20

# Energy density: one-hue blue ramp.  Velocities: blue <-> orange, neutral gray at 0.
# Curves: gray = single events, blue = average over events, orange = the selected event.
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#b4b2a9"
CMAP_E = LinearSegmentedColormap.from_list(
    "e_blue", ["#f4f8fd", "#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]
).with_extremes(bad="#f4f8fd", under="#f4f8fd")
CMAP_V = LinearSegmentedColormap.from_list(
    "v_div", ["#104281", "#3987e5", "#b7d3f6", "#e6e4dc", "#f6c1a8", "#eb6834", "#a8410f"]
).with_extremes(bad="#ffffff")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.edgecolor": "#8a8880",
                     "xtick.color": "#5f5e5a", "ytick.color": "#5f5e5a"})

## 1. Summary

In [ ]:
paths = sorted(glob.glob(FILES))
if not paths:
    raise FileNotFoundError(f"no files match {FILES!r}: set FILES or PROD_H5_GLOB")
files = {p: h5py.File(p, "r") for p in paths}


def axes(f):
    # Cell centres of the output grid [fm], and the tau of every frame [fm/c].
    a, (_, _, nx, ny, neta, ntau) = f.attrs, f["arr"].shape
    ax = lambda lo, d, n: float(a[lo]) + float(a[d]) * np.arange(n)
    return {"x": ax("x_min", "dx", nx), "y": ax("y_min", "dy", ny),
            "eta": ax("eta_min", "deta", neta), "tau": ax("tau_min", "dtau", ntau)}


def max_ntau_setting(f):
    # tau.max_ntau of the grid YAML the job ran with (0 = the tau axis grows).
    try:
        import yaml
        return int((yaml.safe_load(f.attrs["prod_grid_yaml"]).get("tau") or {}).get("max_ntau") or 0)
    except Exception:
        return 0


events = []
for p, f in files.items():
    a, tau = f.attrs, axes(f)["tau"]
    # Unclipped events end at most one MUSIC step + one output step before tau_freezeout.
    gap_ok = float(a["dtau"]) + float(a.get("dtau_MUSIC", a["dtau"])) + 1e-3
    diag = f.get("diag", {})
    for i in range(int(a.get("nevents_written", f["arr"].shape[0]))):
        n, tfo = int(f["ntau_freezeout"][i]), float(f["tau_freezeout"][i])
        tau_last = float(tau[n - 1]) if n > 0 else np.nan
        events.append({
            "file": os.path.basename(p), "event": i, "seed": int(a.get("prod_seed", -1)),
            "ntau": n, "tau_last": tau_last, "tau_fo": tfo,
            "tau0": float(diag["tau0_music"][i]) if "tau0_music" in diag else np.nan,
            "wall_s": float(diag["wall_s"][i]) if "wall_s" in diag else np.nan,
            "cut": bool(n > 0 and tfo - tau_last > gap_ok), "path": p})
ev = pd.DataFrame(events)

f0 = files[paths[0]]
a0, g0 = f0.attrs, axes(f0)
rng = lambda v: f"{v[0]:+g} .. {v[-1]:+g} (n = {len(v)})"
print(f"{len(paths)} file(s), {len(ev)} event(s);  hydro: {a0.get('hydro')};  "
      f"build: {a0.get('prod_build')}")
print(f"grid  x {rng(g0['x'])}  y {rng(g0['y'])}  eta_s {rng(g0['eta'])}  fm")
print(f"      tau from {float(a0['tau_min']):g} in steps of {float(a0['dtau']):g} fm/c;  "
      f"channels {list(a0['feature_names'])};  {a0.get('units')}")

# File-level consistency: FNO4d reads a set of files together only if they share one grid.
problems = []
for p, f in files.items():
    a, name = f.attrs, os.path.basename(p)
    if not a.get("complete", False):
        problems.append(f"{name}: attrs['complete'] is False (job did not finish)")
    if int(a.get("nevents_written", -1)) != f["arr"].shape[0]:
        problems.append(f"{name}: nevents_written {a.get('nevents_written')} != arr.shape[0] "
                        f"{f['arr'].shape[0]}")
    if f["arr"].shape[1:5] != f0["arr"].shape[1:5] or any(
            not np.isclose(float(a[k]), float(a0[k]))
            for k in ("x_min", "dx", "y_min", "dy", "eta_min", "deta", "tau_min", "dtau")):
        problems.append(f"{name}: output grid differs from {os.path.basename(paths[0])}")
bad_tau0 = ev[ev.tau0 > float(a0["tau_min"]) + 1e-6]
if len(bad_tau0):
    problems.append(f"{len(bad_tau0)} event(s) start after tau_min (tau0 > {float(a0['tau_min']):g})")
extents = sorted({f["arr"].shape[5] for f in files.values()})
if len(extents) > 1:
    print(f"note: tau extents differ between files {extents}; MultiH5Array needs one value, "
          "use jetscape.fno_h5_writer.repad_to() before training on them together.")
print("file checks:", "OK" if not problems else "\n  " + "\n  ".join(problems))

ev.drop(columns="path").style.format(
    {"tau_last": "{:.2f}", "tau_fo": "{:.2f}", "tau0": "{:.3f}", "wall_s": "{:.1f}"})

## 2. Sanity scan

One pass over every written frame of the first `MAX_SCAN_EVENTS` events. It also collects the curves
for section 3. `edge ε max` is the largest $\varepsilon$ on the transverse boundary of the output box.
Fluid above freeze-out there ($\gtrsim 0.2$ GeV/fm$^3$) means the fireball is larger than the chosen $x$–$y$ grid.
That is information about the grid choice, not an error.

In [ ]:
def scan_event(e):
    f = files[e["path"]]
    arr, i, n = f["arr"], e["event"], e["ntau"]
    ieta0 = int(np.argmin(np.abs(axes(f)["eta"])))
    curves = {k: np.full(n, np.nan) for k in ("e_mean", "e_mid", "e_max", "vT")}
    res = {"nonfinite": 0, "e<0": 0, "|v|>=1": 0, "edge ε max": 0.0}
    for t in range(n):
        fr = arr[i, :, :, :, :, t]                      # one chunk: (4, nx, ny, neta)
        e_, vx, vy, vz = fr
        res["nonfinite"] += int(np.count_nonzero(~np.isfinite(fr)))
        res["e<0"] += int(np.count_nonzero(e_ < 0))
        res["|v|>=1"] += int(np.count_nonzero(vx * vx + vy * vy + vz * vz >= 1))
        res["edge ε max"] = max(res["edge ε max"], float(e_[[0, -1]].max()),
                                float(e_[:, [0, -1]].max()))
        w = float(e_.sum(dtype=np.float64))
        curves["e_mean"][t] = w / e_.size
        curves["e_mid"][t] = e_[:, :, ieta0].mean()
        curves["e_max"][t] = e_.max()
        curves["vT"][t] = (e_ * np.hypot(vx, vy)).sum(dtype=np.float64) / w if w > 0 else np.nan
    # Frames past ntau_freezeout are never allocated and must read back as exactly zero.
    res["pad nonzero"] = int(np.count_nonzero(arr[i, ..., n:])) if arr.shape[5] > n else 0
    return res, curves


scan = ev.index[: MAX_SCAN_EVENTS or len(ev)]
checks, curves = [], {}
for k in scan:
    res, curves[k] = scan_event(events[k])
    checks.append({"file": ev.file[k], "event": ev.event[k], **res})
checks = pd.DataFrame(checks)
failing = checks[(checks[["nonfinite", "e<0", "|v|>=1", "pad nonzero"]] > 0).any(axis=1)]
print(f"scanned {len(scan)} of {len(ev)} event(s):",
      "all frames finite, ε >= 0, |v| < 1, padding zero" if failing.empty
      else f"{len(failing)} event(s) FAIL, see the table")
checks.style.format({"edge ε max": "{:.3f}"})

## 3. Evolution vs $\tau$

Each event's own frames only, since events freeze out at different $\tau$. The average (blue) is taken over the
events still running at that $\tau$, and drawn while at least half of them are. The max-$\varepsilon$ curve ends
near the freeze-out energy density: MUSIC stops once no cell is above $T_{fo}$. The dashed line marks its value
at the last frame, which the viewer in section 5 uses as the freeze-out contour.

In [ ]:
SEL = 0            # event (row of the summary table) drawn in orange

tau_all = float(a0["tau_min"]) + float(a0["dtau"]) * np.arange(max(extents))
fig, axs = plt.subplots(1, 3, figsize=(12, 3.4), constrained_layout=True)
panels = [("e_mean", r"$\langle\varepsilon\rangle$ over the grid [GeV/fm$^3$]", True),
          ("e_max", r"max $\varepsilon$ [GeV/fm$^3$]", True),
          ("vT", r"$\langle v_T\rangle_\varepsilon$", False)]
for ax, (key, label, log) in zip(axs, panels):
    stack = np.full((len(curves), len(tau_all)), np.nan)
    for r, cv in enumerate(curves.values()):
        c = cv[key]
        stack[r, : len(c)] = c
        ax.plot(tau_all[: len(c)], c, color=GRAY, lw=0.8, zorder=1)
    alive = np.isfinite(stack).sum(axis=0)
    mean = np.where(alive >= max(1, len(curves) / 2),
                    np.nansum(stack, axis=0) / np.maximum(alive, 1), np.nan)
    ax.plot(tau_all, mean, color=BLUE, lw=2, zorder=3, label=f"average ({len(curves)} events)")
    if SEL in curves:
        c = curves[SEL][key]
        ax.plot(tau_all[: len(c)], c, color=ORANGE, lw=1.5, zorder=4,
                label=f"{ev.file[SEL]} #{ev.event[SEL]}")
    ax.set(xlabel=r"$\tau$ [fm/c]", ylabel=label, yscale="log" if log else "linear")
    ax.grid(True, color="#e6e4dc", lw=0.6)
axs[0].plot([], [], color=GRAY, lw=0.8, label="single events")
axs[0].legend(frameon=False)

# Max ε at each event's last frame ~ the freeze-out energy density (upper estimate).
E_FO = float(np.median([cv["e_max"][-1] for cv in curves.values() if len(cv["e_max"])]))
axs[1].axhline(E_FO, color="#5f5e5a", lw=1, ls="--")
axs[1].annotate(f"last frame: {E_FO:.3f}", (tau_all[0], E_FO), xytext=(2, 3),
                textcoords="offset points", color="#5f5e5a")
plt.show()

## 4. Freeze-out time

`tau_freezeout` is when MUSIC stopped (every cell below $T_{fo}$). It comes from MUSIC's own grid, so it is valid even
when the output $\tau$ axis is pinned (`tau.max_ntau > 0`). In that case `ntau_freezeout` is capped and those events
count as *cut*. Right panel: wall time per event against freeze-out time.

In [ ]:
pinned = sorted({max_ntau_setting(f) for f in files.values()} - {0})
if pinned:
    print(f"tau axis pinned at {pinned} frame(s) (tau.max_ntau); {int(ev.cut.sum())} of {len(ev)} "
          "event(s) ran longer and keep only their first frames.")
else:
    print(f"tau axis grows to each event's length; {int(ev.cut.sum())} event(s) cut.")
s = ev.tau_fo
print(f"tau_freezeout: mean {s.mean():.2f}, std {s.std():.2f}, min {s.min():.2f}, "
      f"max {s.max():.2f} fm/c over {len(s)} event(s)")

dt = float(a0.get("dtau_MUSIC", a0["dtau"]))
bins = (np.arange(np.floor(s.min() / dt), np.ceil(s.max() / dt) + 2) - 0.5) * dt
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2), constrained_layout=True)
axs[0].hist(s, bins=bins, color=BLUE, edgecolor="white", linewidth=1.5)
axs[0].set(xlabel=r"$\tau_{fo}$ [fm/c]", ylabel="events")
axs[1].scatter(s, ev.wall_s, s=36, color=BLUE, edgecolor="white", linewidth=1)
axs[1].set(xlabel=r"$\tau_{fo}$ [fm/c]", ylabel="wall time per event [s]")
for ax in axs:
    ax.set_axisbelow(True)
    ax.grid(True, axis="y", color="#e6e4dc", lw=0.6)
plt.show()

## 5. $x$–$y$ viewer

$\varepsilon$ and the three lab-frame velocities at one $\eta_s$ slice. Each velocity panel has its own symmetric colour
scale. Away from midrapidity $v_z \approx \tanh\eta_s$ everywhere, so *v_z − tanh η_s* shows the deviation from
Bjorken flow instead. Near-vacuum cells outside the fireball carry noisy velocities and are blanked (white) by
default. Arrows show $(v_x, v_y)$, and the dashed contour is $\varepsilon$ = `E_FO` from section 3 (an estimate of
freeze-out). The $\tau$ slider only offers the selected event's
written frames. ▶ plays through $\tau$.

Widgets need a live kernel, so the static strip in the next cell shows the same event in a saved notebook.

In [ ]:
import ipywidgets as W
from IPython.display import display

LABELS = [r"$\varepsilon$ [GeV/fm$^3$]", r"$v_x$", r"$v_y$", r"$v_z$"]


def extent(g):
    dx, dy = g["x"][1] - g["x"][0], g["y"][1] - g["y"][0]
    return [g["x"][0] - dx / 2, g["x"][-1] + dx / 2, g["y"][0] - dy / 2, g["y"][-1] + dy / 2]


def draw_xy(iev=0, itau=0, ieta=0, log_e=True, arrows=True, contour=True, mask_v=True,
            dvz=False):
    e = events[iev]
    f = files[e["path"]]
    g = axes(f)
    itau, ieta = min(itau, e["ntau"] - 1), min(ieta, len(g["eta"]) - 1)
    fr = f["arr"][e["event"], :, :, :, ieta, itau]          # (4, nx, ny); x is axis 0
    if dvz:
        fr[3] -= np.tanh(g["eta"][ieta])                      # deviation from Bjorken flow
    if mask_v:                                                # near-vacuum cells: noisy v
        fr[1:, fr[0] < E_FO / 10] = np.nan
    ext = extent(g)
    fig, axs = plt.subplots(2, 2, figsize=(9.5, 8), constrained_layout=True)
    for k, ax in enumerate(axs.flat):
        z = fr[k].T                                           # imshow rows are y
        if k == 0:
            zmax = float(z.max())
            norm = (LogNorm(vmin=max(zmax * 1e-3, 1e-5), vmax=zmax) if log_e and zmax > 0
                    else Normalize(0, max(zmax, 1e-9)))
            im = ax.imshow(np.where(z > 0, z, np.nan) if log_e else z, origin="lower",
                           extent=ext, cmap=CMAP_E, norm=norm)
            if contour and z.min() < E_FO < zmax:
                ax.contour(g["x"], g["y"], z, levels=[E_FO], colors="#0d366b",
                           linewidths=1, linestyles="--")
            if arrows:
                s = max(1, len(g["x"]) // 16)
                vx, vy = fr[1, ::s, ::s].T, fr[2, ::s, ::s].T
                keep = fr[0, ::s, ::s].T > (E_FO / 4)
                ax.quiver(g["x"][::s], g["y"][::s], np.where(keep, vx, np.nan),
                          np.where(keep, vy, np.nan), color=ORANGE, scale=12, width=0.004)
        else:
            vm = max(float(np.nanmax(np.abs(z), initial=0)), 1e-6)
            im = ax.imshow(z, origin="lower", extent=ext, cmap=CMAP_V, vmin=-vm, vmax=vm)
        fig.colorbar(im, ax=ax, shrink=0.9,
                     label=r"$v_z - \tanh\eta_s$" if k == 3 and dvz else LABELS[k])
        ax.set(xlabel="x [fm]", ylabel="y [fm]", aspect="equal")
    fig.suptitle(f"{e['file']}  event {e['event']}   τ = {g['tau'][itau]:.2f} fm/c   "
                 f"η_s = {g['eta'][ieta]:+.3f}   (last frame {e['tau_last']:.2f}, "
                 f"τ_fo {e['tau_fo']:.2f})")
    plt.show()


def tau_options(e):
    return [(f"{t:.2f}", i) for i, t in enumerate(axes(files[e["path"]])["tau"][: e["ntau"]])]


eta0 = [(f"{x:+.3f}", j) for j, x in enumerate(g0["eta"])]
w_ev = W.Dropdown(options=[(f"{e['file']} #{e['event']}  ({e['ntau']} frames)", k)
                           for k, e in enumerate(events)], value=0, description="event",
                  layout=W.Layout(width="420px"))
w_tau = W.SelectionSlider(options=tau_options(events[0]), value=0, description="τ [fm/c]",
                          continuous_update=False, layout=W.Layout(width="620px"))
w_eta = W.SelectionSlider(options=eta0, value=int(np.argmin(np.abs(g0["eta"]))),
                          description="η_s", continuous_update=False,
                          layout=W.Layout(width="620px"))
w_log = W.Checkbox(value=True, description="log ε", indent=False)
w_arr = W.Checkbox(value=True, description="flow arrows", indent=False)
w_con = W.Checkbox(value=True, description="ε_fo contour", indent=False)
w_msk = W.Checkbox(value=True, description="hide v where ε < ε_fo/10", indent=False)
w_dvz = W.Checkbox(value=False, description="v_z − tanh η_s", indent=False)
w_play = W.Play(min=0, max=len(w_tau.options) - 1, interval=400)
W.jslink((w_play, "value"), (w_tau, "index"))


def on_event(change):
    e = events[change["new"]]
    old = w_tau.index
    w_tau.options = tau_options(e)
    w_tau.index = min(old, len(w_tau.options) - 1)
    w_play.max = len(w_tau.options) - 1
    w_eta.options = [(f"{x:+.3f}", j) for j, x in enumerate(axes(files[e["path"]])["eta"])]


w_ev.observe(on_event, names="value")
out = W.interactive_output(draw_xy, {"iev": w_ev, "itau": w_tau, "ieta": w_eta,
                                     "log_e": w_log, "arrows": w_arr, "contour": w_con,
                                     "mask_v": w_msk, "dvz": w_dvz})
display(W.VBox([W.HBox([w_ev, w_log, w_arr, w_con]), W.HBox([w_msk, w_dvz]),
                W.HBox([w_play, w_tau]), w_eta]), out)

In [ ]:
# Static strip: ε at the η_s slice nearest 0 for event SEL at six τ, one shared log scale.
e = events[SEL]
f, g = files[e["path"]], axes(files[e["path"]])
ieta = int(np.argmin(np.abs(g["eta"])))
its = np.unique(np.linspace(0, e["ntau"] - 1, 6).round().astype(int))
frames = [f["arr"][e["event"], 0, :, :, ieta, t].T for t in its]
vmax = max(float(z.max()) for z in frames)
fig, axs = plt.subplots(1, len(its), figsize=(2.3 * len(its) + 1, 2.8), constrained_layout=True)
for ax, t, z in zip(np.atleast_1d(axs), its, frames):
    im = ax.imshow(np.where(z > 0, z, np.nan), origin="lower", extent=extent(g), cmap=CMAP_E,
                   norm=LogNorm(vmin=max(E_FO / 10, 1e-5), vmax=vmax))
    if z.max() > E_FO:
        ax.contour(g["x"], g["y"], z, levels=[E_FO], colors="#0d366b", linewidths=0.8,
                   linestyles="--")
    ax.set_title(f"τ = {g['tau'][t]:.1f} fm/c")
    ax.set(xlabel="x [fm]", aspect="equal")
np.atleast_1d(axs)[0].set_ylabel("y [fm]")
fig.colorbar(im, ax=axs, shrink=0.9, label=r"$\varepsilon$ [GeV/fm$^3$]")
fig.suptitle(f"{e['file']} event {e['event']},  η_s = {g['eta'][ieta]:+.3f}")
plt.show()